# Getting started with AudioSeal

This notebook shows a minimal example to get started with AudioSeal. Make sure you install the package from PyPi or from source in editable mode beforehand.

First, let's prepare some example audios into the proper Tensor format using Torchaudio

In [1]:
%%capture

# For the demonstration, we need torchaudio and matplotlib to process example audios and visualize the spectrogram
import sys
!{sys.executable} -m pip install torchaudio soundfile matplotlib

In [2]:
import torch
import torchaudio
import urllib

## Generator

To watermark an audio, we simply load the watermarking generator from the hub:

In [3]:
import sys
sys.path.append("..")  # 添加上一级目录
from src.audioseal import AudioSeal

model = AudioSeal.load_generator("audioseal_wm_16bits")

### Watermarking with a secret message:

AudioSeal uses a secret message to generate a watermarking, which consists of <i>n</i> binary bits (n=16 for the above model, hence the model name). By default, if the user does not specify these n bits, a random message will be generated. This can be customized either by assigning the secret message before the watermarking process:

```
`model.message = <n bit Tensor>`
``` 

or by executing the model each time with one explicit secret message:

```
watermark = model(audio, message=<message>)
```

In [4]:
# watermarked_audio = model(audios, sample_rate=sr, message=secret_mesage, alpha=1)

## generate my watermark seal

In [5]:
import soundfile
import os 
data_path = './my_choose_test_50'
secret_dir = './my_secret_50'
save_dir = './audio_seal_50'
my_audio = os.listdir(data_path)
for i in my_audio:
    wav, sr = torchaudio.load(f'{data_path}/{i}')
    print(sr)
    secret_mesage = torch.randint(0, 2, (1, 16), dtype=torch.int32)
    watermarked_audio = model(wav.unsqueeze(0), sample_rate=sr, message=secret_mesage, alpha=1)

    torch.save(secret_mesage, f"{secret_dir}/{i}_msg_tensor.pt")
    soundfile.write(f'{save_dir}/audio_seal_{i}', watermarked_audio.cpu().squeeze(0).squeeze(0).detach().numpy(), samplerate=sr)

16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000
16000


In [7]:
acc = []
detector = AudioSeal.load_detector(("audioseal_detector_16bits"))

data_path = './my_choose_test_50'
my_seal_audio = os.listdir(data_path)

for i in my_seal_audio:
    wav, sr = torchaudio.load(f'{data_path}/{i}')

    secret_mesage = torch.load(f'./{secret_dir}/{i.split("_")[-1]}_msg_tensor.pt')
    result, message = detector.detect_watermark(wav.unsqueeze(0), sample_rate=sr, message_threshold=0.5)
    acc.append(result)

acc

/tmp/ipykernel_44684/461761257.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  secret_mesage = torch.load(f'./{secret_dir}/{i.split("_")[-1]}_msg_tensor.pt')


[0.0008778744377195835,
 0.008452150970697403,
 0.0,
 0.0,
 0.0,
 0.0,
 0.011500000022351742,
 0.0,
 0.0022236842196434736,
 0.001752419164404273,
 0.03435185179114342,
 0.0,
 0.0,
 0.004334965255111456,
 0.0,
 0.0,
 0.0,
 0.012918307445943356,
 0.0,
 0.0,
 0.0,
 0.0016007532831281424,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0016213388880714774,
 0.0010685011511668563,
 0.0014470622409135103,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.004081347957253456,
 0.0,
 0.0,
 0.00034340660204179585,
 0.009214092046022415,
 0.004712207242846489,
 0.0012719742953777313,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0]

In [8]:
sum(acc)/50

0.0020354387402767315